# 01 · Unión auditada de los datos crudos

Genera la matriz común anterior al endpoint y a M-ComBat, y elimina ya aquí las cohortes con menos de dos etiquetas. No armoniza ni divide cohortes.

In [ ]:
from pathlib import Path
import os, sys

DATA_PREPARE = Path.cwd().resolve()
while not (DATA_PREPARE / "combat_ref.py").exists():
    if DATA_PREPARE.parent == DATA_PREPARE:
        raise FileNotFoundError("Ejecuta este notebook dentro de data_prepare/.")
    DATA_PREPARE = DATA_PREPARE.parent
os.chdir(DATA_PREPARE)
if str(DATA_PREPARE) not in sys.path:
    sys.path.insert(0, str(DATA_PREPARE))
print("DATA_PREPARE:", DATA_PREPARE)

In [ ]:
import pandas as pd
from union_datasets import ConfigUnion, unir

cfg = ConfigUnion(
    micro_matrix=str(DATA_PREPARE / "Data/expr_genes_microarrays.csv"),
    micro_meta=str(DATA_PREPARE / "Data/microarrays_meta.csv"),
    tcga_matrix=str(DATA_PREPARE / "Data/counts_symbol_TCGA.csv"),
    tcga_meta=str(DATA_PREPARE / "Data/TCGA_meta.csv"),
    gse_matrix=str(DATA_PREPARE / "Data/counts_symbol_GSE96058.csv"),
    gse_meta=str(DATA_PREPARE / "Data/GSE96058_meta.csv"),
    gene_mode="interseccion",
    out_dir=str(DATA_PREPARE / "output_union"),
)
res = unir(cfg)

In [ ]:
display(res["escalas"])
display(res["cobertura"])
display(res["cohort_audit"])
display(pd.crosstab(res["meta"]["dataset"], res["meta"]["label"]))
if len(res["duplicados"]):
    raise RuntimeError("Hay posibles duplicados entre cohortes; revisa output_union antes de continuar.")

EXPECTED_CLASSES = {"LumA", "LumB", "HER2-enriched", "Basal-like"}
EXPECTED_COHORTS = {
    "GSE1456", "GSE162228", "GSE20711", "GSE21653", "GSE25066",
    "GSE96058", "TCGA-BRCA",
}
assert set(res["meta"]["label"]) == EXPECTED_CLASSES
assert set(res["meta"]["dataset"]) == EXPECTED_COHORTS
assert not res["meta"]["label"].isin({"Normal", "HER2", "TNBC"}).any()
print("✓ contrato intrinsic4 validado en la unión")
